# 🚀 rocket-env — Colab 학습 노트북

강화학습으로 로켓을 착륙/포획시킨다. 이 노트북 하나로 **설치 → 학습 → 평가 → 영상**까지 된다.

- 런타임 유형은 **CPU**로 충분하다(MlpPolicy는 작다). GPU여도 무방.
- 값을 바꿔 가며 **신경망 구조·하이퍼파라미터·알고리즘**을 직접 탐험하는 게 과제의 핵심이다.


## 1. 설치

환경(`rocket_env`)과 학습 라이브러리(stable-baselines3), 영상 인코더를 설치한다.


In [ ]:
# ── 설치: 아래 셋 중 하나 (조교 안내에 따라) ─────────────────────────

# [A] 저장소가 public 인 경우 — 한 줄로 끝
!pip -q install "rocket-env[sb3] @ git+https://github.com/essohn/rocket-env.git" imageio imageio-ffmpeg

# [B] private + 읽기전용 토큰(조교 발급) — <TOKEN> 을 바꿔서
# !pip -q install "rocket-env[sb3] @ git+https://<TOKEN>@github.com/essohn/rocket-env.git" imageio imageio-ffmpeg

# [C] private — 조교가 준 wheel(.whl) 파일로 설치
#   1) 좌측 파일창에 rocket_env-0.1.0-py3-none-any.whl 업로드(또는 Drive 마운트)
#   2) 아래 실행
# !pip -q install "rocket_env-0.1.0-py3-none-any.whl[sb3]" imageio imageio-ffmpeg

print('설치 완료')

## 2. 헤드리스 렌더 설정

Colab은 화면이 없으므로 더미 비디오 드라이버를 쓴다(**첫 pygame import 전에** 실행).


In [ ]:
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"

import numpy as np
import gymnasium as gym
import rocket_env  # 환경 등록
from rocket_env.config import PRESETS
print('라운드:', list(PRESETS))


## 3. 동작 확인 (무작위 정책)

환경이 도는지 무작위 행동으로 한 판 돌려본다.


In [ ]:
env = gym.make("rocket-v0", config=PRESETS["landing-basic"])
obs, info = env.reset(seed=0)
done = trunc = False
while not (done or trunc):
    obs, r, done, trunc, info = env.step(env.action_space.sample())
print("결과:", info["outcome"], " 성공:", info["is_success"], " 접지속도:", info["impact_speed"])
env.close()


## 4. 학습

아래는 **일부러 최소한의 기본값**이다. 이대로면 basic도 어중간하게만 학습된다.
성능을 끌어올리는 건 여러분 몫이다 — 아래 `# 탐험` 주석의 축들을 바꿔 보라.

> 팁: 라운드가 올라갈수록(attitude 이상) 바닐라 DQN은 벽에 부딪힌다. 왜인지,
> 무엇을 바꿔야 넘는지 스스로 찾아보라.


In [ ]:
from stable_baselines3 import DQN

PRESET = "landing-basic"   # 탐험: 라운드를 바꿔 보라
env = gym.make('rocket-v0', config=PRESETS[PRESET])

model = DQN(
    "MlpPolicy", env, verbose=0, device="cpu",
    # 탐험: 아래를 바꿔 성능이 어떻게 달라지는지 관찰하라
    # policy_kwargs={'net_arch': [64, 64]},   # 신경망 구조(층 크기)
    # learning_rate=1e-4,                      # 학습률
    # gamma=0.99,                              # 할인율
    # buffer_size=100_000, batch_size=32,      # 리플레이
)
model.learn(total_timesteps=200_000)   # 탐험: 더 오래 학습하면?
print('학습 완료')


## 5. 평가

고정 시드 30판을 결정론적으로 돌려 성공률과 평균 접지속도를 잰다.


In [ ]:
import math
def evaluate(model, cfg, n=30, normalize_obs=None):
    env = gym.make('rocket-v0', config=cfg)
    wins, speeds = 0, []
    for i in range(n):
        obs, _ = env.reset(seed=20000 + i)
        while True:
            o = normalize_obs(obs) if normalize_obs else obs
            a, _ = model.predict(o, deterministic=True)
            obs, _, term, trunc, info = env.step(int(a))
            if term or trunc: break
        wins += int(info['is_success'])
        st = env.unwrapped.state; speeds.append(math.hypot(st.vx, st.vy))
    env.close()
    print(f'성공률 {wins}/{n} ({wins/n*100:.0f}%)  평균 접지속도 {sum(speeds)/len(speeds):.1f} m/s')
    return wins / n

evaluate(model, PRESETS[PRESET])


## 6. 영상으로 보기

가장 잘한 에피소드를 골라 mp4로 인코딩해 노트북에 띄운다.


In [ ]:
import imageio
from IPython.display import HTML
from base64 import b64encode

def record(model, cfg, n=20, normalize_obs=None, out='demo.mp4'):
    env = gym.make('rocket-v0', config=cfg, render_mode='rgb_array')
    best = None
    for i in range(n):
        obs, _ = env.reset(seed=10000 + i)
        frames = [env.render()]
        while True:
            o = normalize_obs(obs) if normalize_obs else obs
            a, _ = model.predict(o, deterministic=True)
            obs, _, term, trunc, info = env.step(int(a))
            frames.append(env.render())
            if term or trunc: break
        key = (info['is_success'], -(info['impact_speed'] or 1e9))
        if best is None or key > best[0]:
            best = (key, frames, info)
    env.close()
    imageio.mimsave(out, best[1], fps=20)
    print('선택 에피소드:', best[2]['outcome'], '성공' if best[2]['is_success'] else '실패')
    return out

path = record(model, PRESETS[PRESET])
mp4 = b64encode(open(path,'rb').read()).decode()
HTML(f'<video width=320 controls><source src="data:video/mp4;base64,{mp4}" type="video/mp4"></video>')


## 7. 상위 라운드 — PPO + 정규화

고고도 라운드(descent 이상)는 바닐라 DQN으로 잘 안 된다. 더 강한 셋업이 필요하다.
아래는 **참고 골격**이다. ⚠️ 정규화를 쓰면 통계를 모델과 함께 저장/복원해야 한다.


In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

cfg = PRESETS['landing-descent']
venv = VecNormalize(DummyVecEnv([lambda: gym.make('rocket-v0', config=cfg)]),
                    norm_obs=True, norm_reward=True, clip_obs=10.0)
ppo = PPO('MlpPolicy', venv, verbose=0, device='cpu',
          # 탐험: gamma, ent_coef, n_steps, net_arch ...
          )
ppo.learn(total_timesteps=1_000_000)

# 평가·저장 시엔 통계를 고정하고 보상 정규화를 끈다
venv.training = False; venv.norm_reward = False
evaluate(ppo, cfg, normalize_obs=venv.normalize_obs)


## 8. 모델 저장 / 내려받기

제출용으로 모델을 저장한다. PPO+정규화는 통계(`.pkl`)도 함께 필요하다.


In [ ]:
model.save('dqn_landing_basic.zip')
# PPO 를 저장한다면:
# ppo.save('ppo_descent.zip'); venv.save('ppo_descent-vecnorm.pkl')

from google.colab import files
files.download('dqn_landing_basic.zip')
